# ADAS project: lightweight vehicle detectors on KITTI (main notebook)
**Comparative Benchmarking of Lightweight Object Detectors for Real-Time Vehicle Detection on Edge Devices**

This notebook handles **YOLOv8n, YOLOv9-tiny and SSDLite-MobileNetV3**. NanoDet needs an older software stack, so it has its own notebook (`adas_nanodet_colab.ipynb`). Both write the same result files, and the last cells of this notebook merge everything into the final comparison table, plots and feasibility analysis.

**Fairness rules built in:** same KITTI split (Ultralytics' copy), same input size, **one common evaluator** (torchmetrics mAP) for every model, same latency method (batch size 1, warm-up excluded, pre/post-processing included).

**How to run:** `Runtime > Change runtime type > T4 GPU`, then run the cells top to bottom. A full run takes several hours, so in a team of 4 set `RUN_MODELS` in the config cell to just *your* model(s), then collect everyone's `result_*.json` files into one `adas_benchmark/results` folder and run the last cells.

| Member | `RUN_MODELS` |
|---|---|
| A | `["yolov8n"]` |
| B | `["yolov9t"]` |
| C | `["ssdlite"]` |
| D | run `adas_nanodet_colab.ipynb` |

First do a test: set `QUICK = True` (about 15 minutes, tiny subset, 1 epoch) to confirm that everything runs, then set it back to `False`.

## 1. Setup

In [ ]:
!pip -q install ultralytics torchmetrics pycocotools pandas
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU found. Switch the runtime to T4 GPU.")

In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/adas_benchmark")
except Exception as e:
    print("Drive not mounted, saving locally:", e)
    OUT = Path("/content/adas_benchmark")
RES, WTS, FIG = OUT / "results", OUT / "weights", OUT / "figures"
for p in (RES, WTS, FIG):
    p.mkdir(parents=True, exist_ok=True)
print("Saving everything to:", OUT)

In [ ]:
# ----------------------------- CONFIG -----------------------------
RUN_MODELS = ["yolov8n", "yolov9t", "ssdlite"]   # team: keep only YOUR model here
QUICK = False        # True = tiny smoke test (1 epoch, small subset)
FORCE = False        # True = redo a model even if its result file exists

DATA = "kitti.yaml"  # Ultralytics downloads KITTI automatically
IMGSZ = 640          # YOLO input size
SSD_SIZE = 640       # SSDLite input size (its native size is 320; 640 matches the YOLO models)
EPOCHS = {"yolov8n": 30, "yolov9t": 30, "ssdlite": 20}
BATCH = {"yolov8n": 16, "yolov9t": 16, "ssdlite": 16}
SEED = 0
MAX_TRAIN, MAX_VAL = (200, 100) if QUICK else (None, None)
if QUICK:
    EPOCHS = {k: 1 for k in EPOCHS}
import os
CPU_THREADS = min(4, os.cpu_count() or 1)   # CPU threads used as the edge-device stand-in
NAMES = {"yolov8n": "YOLOv8n", "yolov9t": "YOLOv9-tiny", "ssdlite": "SSDLite-MobileNetV3", "nanodet": "NanoDet-Plus-m"}

In [ ]:
import os, io, time, json, math, random, shutil, contextlib
from pathlib import Path
import numpy as np, cv2, torch
from tqdm.auto import tqdm
from torchmetrics.detection.mean_ap import MeanAveragePrecision

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")
VEHICLE_NAMES = {"car", "van", "truck", "tram", "bus"}   # classes counted as "vehicles"

def list_images(p):
    if isinstance(p, (list, tuple)):
        p = p[0]
    return sorted(str(x) for x in Path(p).rglob("*") if x.suffix.lower() in IMG_EXT)

def label_path(img_path):
    a, b = str(img_path).rsplit(os.sep + "images" + os.sep, 1)
    return Path(a, "labels", b).with_suffix(".txt")

def read_targets(img_path, w, h):
    """YOLO txt labels -> boxes in pixel xyxy + class ids."""
    boxes, labels = [], []
    lp = label_path(img_path)
    if lp.exists():
        for line in lp.read_text().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            c = int(float(parts[0]))
            cx, cy, bw, bh = [float(v) for v in parts[1:5]]
            boxes.append([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h])
            labels.append(c)
    return np.array(boxes, dtype=np.float32).reshape(-1, 4), np.array(labels, dtype=np.int64)

def yolo_to_coco(img_list, names, root, out_json):
    images, anns, aid = [], [], 1
    for i, p in enumerate(img_list, 1):
        im = cv2.imread(p)
        h, w = im.shape[:2]
        images.append({"id": i, "file_name": os.path.relpath(p, root), "width": w, "height": h})
        b, l = read_targets(p, w, h)
        for (x1, y1, x2, y2), c in zip(b, l):
            bw, bh = float(x2 - x1), float(y2 - y1)
            if bw < 1 or bh < 1:
                continue
            anns.append({"id": aid, "image_id": i, "category_id": int(c) + 1,
                         "bbox": [float(x1), float(y1), bw, bh], "area": bw * bh, "iscrowd": 0})
            aid += 1
    cats = [{"id": k + 1, "name": v} for k, v in sorted(names.items())]
    with open(out_json, "w") as f:
        json.dump({"images": images, "annotations": anns, "categories": cats}, f)

def _t(a, dtype):
    return torch.as_tensor(np.asarray(a, dtype=dtype))

def evaluate_model(predict_fn, img_list, names):
    """ONE evaluator for every model: same images, same torchmetrics mAP.
    predict_fn(bgr_image) -> (boxes_xyxy, scores, class_ids)."""
    vid = torch.tensor([i for i, n in names.items() if n in VEHICLE_NAMES], dtype=torch.int64)
    m_all = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
    m_veh = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
    for p in tqdm(img_list, desc="evaluating"):
        img = cv2.imread(p)
        h, w = img.shape[:2]
        gb, gl = read_targets(p, w, h)
        pb, ps, pl = predict_fn(img)
        tgt = {"boxes": _t(gb, np.float32).reshape(-1, 4), "labels": _t(gl, np.int64)}
        prd = {"boxes": _t(pb, np.float32).reshape(-1, 4), "scores": _t(ps, np.float32), "labels": _t(pl, np.int64)}
        m_all.update([prd], [tgt])
        gk, pk = torch.isin(tgt["labels"], vid), torch.isin(prd["labels"], vid)
        m_veh.update([{k: v[pk] for k, v in prd.items()}], [{k: v[gk] for k, v in tgt.items()}])
    ra, rv = m_all.compute(), m_veh.compute()
    return {"map50": float(ra["map_50"]), "map": float(ra["map"]),
            "map50_vehicle": float(rv["map_50"]), "map_vehicle": float(rv["map"])}

def measure_latency(predict_fn, img_list, cuda, n=100, warmup=10):
    """End-to-end latency per image (pre-processing + model + post-processing), batch size 1."""
    pick = random.Random(0).sample(img_list, min(n + warmup, len(img_list)))
    imgs = [cv2.imread(p) for p in pick]
    for im in imgs[:warmup]:
        predict_fn(im)
    lat = []
    for im in imgs[warmup:]:
        if cuda:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        predict_fn(im)
        if cuda:
            torch.cuda.synchronize()
        lat.append((time.perf_counter() - t0) * 1000)
    lat = np.array(lat)
    return {"mean_ms": float(lat.mean()), "p95_ms": float(np.percentile(lat, 95)), "fps": float(1000 / lat.mean())}

def save_result(res_dir, tag, d):
    Path(res_dir).mkdir(parents=True, exist_ok=True)
    with open(Path(res_dir) / f"result_{tag}.json", "w") as f:
        json.dump(d, f, indent=2)
    return d

## 2. Dataset (KITTI)

In [ ]:
from ultralytics.data.utils import check_det_dataset
d = check_det_dataset(DATA)
names = {int(k): v for k, v in d["names"].items()} if isinstance(d["names"], dict) else dict(enumerate(d["names"]))
NC = len(names)
train_imgs, val_imgs = list_images(d["train"]), list_images(d["val"])
random.Random(SEED).shuffle(train_imgs)
if MAX_TRAIN:
    train_imgs, val_imgs = train_imgs[:MAX_TRAIN], val_imgs[:MAX_VAL]
print(f"classes ({NC}):", names)
print("train images:", len(train_imgs), "| val images:", len(val_imgs))
print("vehicle classes:", [n for n in names.values() if n in VEHICLE_NAMES])

## 3. YOLOv8n and YOLOv9-tiny (Ultralytics)

In [ ]:
from ultralytics import YOLO

def make_yolo_predictor(weights, device, conf, imgsz=IMGSZ):
    mdl = YOLO(str(weights))
    def fn(img):
        r = mdl.predict(img, imgsz=imgsz, conf=conf, iou=0.6, device=device, verbose=False)[0]
        b = r.boxes
        return b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy().astype(int)
    return fn

def run_yolo(tag, weights):
    res_file = RES / f"result_{tag}.json"
    if res_file.exists() and not FORCE:
        print("Already done, skipping:", res_file.name)
        return json.loads(res_file.read_text())
    model = YOLO(weights)
    extra = {"fraction": 0.05} if QUICK else {}
    model.train(data=DATA, epochs=EPOCHS[tag], imgsz=IMGSZ, batch=BATCH[tag], seed=SEED,
                project=str(OUT / "runs"), name=tag, exist_ok=True, patience=10, plots=True, **extra)
    run_dir = Path(model.trainer.save_dir)
    best_pt = WTS / f"{tag}.pt"
    shutil.copy(run_dir / "weights" / "best.pt", best_pt)
    for f in ("results.png", "confusion_matrix_normalized.png", "PR_curve.png"):
        if (run_dir / f).exists():
            shutil.copy(run_dir / f, FIG / f"{tag}_{f}")

    cuda = torch.cuda.is_available()
    ev = evaluate_model(make_yolo_predictor(best_pt, 0 if cuda else "cpu", 0.001), val_imgs, names)
    gpu = measure_latency(make_yolo_predictor(best_pt, 0, 0.25), val_imgs, True) if cuda else None
    torch.set_num_threads(CPU_THREADS)
    cpu = measure_latency(make_yolo_predictor(best_pt, "cpu", 0.25), val_imgs, False)
    m = YOLO(str(best_pt))
    onnx_mb = None
    try:
        onnx_path = m.export(format="onnx", imgsz=IMGSZ, simplify=True)
        onnx_mb = os.path.getsize(onnx_path) / 1e6
    except Exception as e:
        print("ONNX export skipped:", e)
    out = {"tag": tag, "model": NAMES[tag], "input_size": IMGSZ, **ev,
           "params_m": sum(p.numel() for p in m.model.parameters()) / 1e6,
           "size_mb": os.path.getsize(best_pt) / 1e6, "onnx_mb": onnx_mb,
           "gpu_ms": gpu and gpu["mean_ms"], "gpu_fps": gpu and gpu["fps"],
           "cpu_ms": cpu["mean_ms"], "cpu_fps": cpu["fps"], "cpu_p95_ms": cpu["p95_ms"], "cpu_threads": CPU_THREADS,
           "epochs": EPOCHS[tag]}
    save_result(RES, tag, out)
    print(json.dumps(out, indent=2))
    return out

In [ ]:
if "yolov8n" in RUN_MODELS:
    r_v8 = run_yolo("yolov8n", "yolov8n.pt")

In [ ]:
if "yolov9t" in RUN_MODELS:
    r_v9 = run_yolo("yolov9t", "yolov9t.pt")

## 4. SSDLite-MobileNetV3 (torchvision)

In [ ]:
from torchvision.models.detection import ssdlite320_mobilenet_v3_large

def build_ssd(nc, size, pretrained=True):
    """COCO-pretrained SSDLite; the classification layers are re-initialised for our classes."""
    model = ssdlite320_mobilenet_v3_large(weights=None, weights_backbone=None, num_classes=nc + 1)
    if pretrained:
        coco = ssdlite320_mobilenet_v3_large(weights="DEFAULT")
        sd = model.state_dict()
        keep = {k: v for k, v in coco.state_dict().items() if k in sd and v.shape == sd[k].shape}
        model.load_state_dict(keep, strict=False)
        print(f"loaded {len(keep)}/{len(sd)} pretrained tensors")
    model.transform.fixed_size = (size, size)
    return model

def make_ssd_predictor(model, device, conf):
    model = model.to(device).eval()
    model.score_thresh, model.detections_per_img = conf, 100
    @torch.no_grad()
    def fn(img):
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        t = torch.from_numpy(rgb).permute(2, 0, 1).float().div(255).to(device)
        o = model([t])[0]
        return o["boxes"].cpu().numpy(), o["scores"].cpu().numpy(), o["labels"].cpu().numpy() - 1
    return fn

class YoloDet(torch.utils.data.Dataset):
    def __init__(self, imgs, train):
        self.imgs, self.train = imgs, train
    def __len__(self):
        return len(self.imgs)
    def __getitem__(self, i):
        p = self.imgs[i]
        img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        b, l = read_targets(p, w, h)
        t = torch.from_numpy(img).permute(2, 0, 1).float().div(255)
        b = torch.as_tensor(b, dtype=torch.float32).reshape(-1, 4)
        l = torch.as_tensor(l, dtype=torch.int64) + 1          # 0 is background in torchvision
        if self.train and random.random() < 0.5:                # horizontal flip
            t = t.flip(-1)
            b = torch.stack([w - b[:, 2], b[:, 1], w - b[:, 0], b[:, 3]], 1)
        keep = ((b[:, 2] - b[:, 0]) > 1) & ((b[:, 3] - b[:, 1]) > 1)
        return t, {"boxes": b[keep], "labels": l[keep]}

def collate(batch):
    return tuple(zip(*batch))

def run_ssd(tag="ssdlite"):
    res_file = RES / f"result_{tag}.json"
    if res_file.exists() and not FORCE:
        print("Already done, skipping:", res_file.name)
        return json.loads(res_file.read_text())
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(SEED)
    model = build_ssd(NC, SSD_SIZE).to(dev)
    dl = torch.utils.data.DataLoader(YoloDet(train_imgs, True), batch_size=BATCH[tag], shuffle=True,
                                     num_workers=2, collate_fn=collate, drop_last=True)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=0.01, momentum=0.9, weight_decay=4e-5)
    total = EPOCHS[tag] * len(dl)
    warm = max(1, min(300, total // 5))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda it: (it + 1) / warm if it < warm
                                              else 0.5 * (1 + math.cos(math.pi * (it - warm) / max(1, total - warm))))
    for ep in range(EPOCHS[tag]):
        model.train()
        run_loss = 0.0
        for imgs, tgts in tqdm(dl, desc=f"epoch {ep + 1}/{EPOCHS[tag]}"):
            imgs = [i.to(dev) for i in imgs]
            tgts = [{k: v.to(dev) for k, v in t.items()} for t in tgts]
            loss = sum(model(imgs, tgts).values())
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 10.0)
            opt.step()
            sched.step()
            run_loss += float(loss)
        print(f"epoch {ep + 1}: mean loss {run_loss / len(dl):.4f}")
        torch.save(model.state_dict(), WTS / f"{tag}.pt")
    cuda = torch.cuda.is_available()
    ev = evaluate_model(make_ssd_predictor(model, dev, 0.001), val_imgs, names)
    gpu = measure_latency(make_ssd_predictor(model, "cuda", 0.25), val_imgs, True) if cuda else None
    torch.set_num_threads(CPU_THREADS)
    cpu_model = build_ssd(NC, SSD_SIZE, pretrained=False)
    cpu_model.load_state_dict(torch.load(WTS / f"{tag}.pt", map_location="cpu"))
    cpu = measure_latency(make_ssd_predictor(cpu_model, "cpu", 0.25), val_imgs, False)
    out = {"tag": tag, "model": NAMES[tag], "input_size": SSD_SIZE, **ev,
           "params_m": sum(p.numel() for p in model.parameters()) / 1e6,
           "size_mb": os.path.getsize(WTS / f"{tag}.pt") / 1e6, "onnx_mb": None,
           "gpu_ms": gpu and gpu["mean_ms"], "gpu_fps": gpu and gpu["fps"],
           "cpu_ms": cpu["mean_ms"], "cpu_fps": cpu["fps"], "cpu_p95_ms": cpu["p95_ms"], "cpu_threads": CPU_THREADS,
           "epochs": EPOCHS[tag]}
    save_result(RES, tag, out)
    print(json.dumps(out, indent=2))
    return out

In [ ]:
if "ssdlite" in RUN_MODELS:
    r_ssd = run_ssd("ssdlite")

## 5. Final comparison (merges every `result_*.json`, including NanoDet's)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
rows = [json.loads(p.read_text()) for p in sorted(RES.glob("result_*.json"))]
assert rows, "No result files found in " + str(RES)
df = pd.DataFrame(rows)
cols = ["model", "input_size", "map50_vehicle", "map_vehicle", "map50", "map", "params_m", "size_mb",
        "gpu_ms", "gpu_fps", "cpu_ms", "cpu_fps", "cpu_threads", "onnx_mb", "epochs"]
table = df[[c for c in cols if c in df.columns]].round(3)
table.to_csv(OUT / "comparison_table.csv", index=False)
display(table)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
for _, r in df.iterrows():
    ax.scatter(r["cpu_fps"], r["map50_vehicle"], s=60 + 25 * r["size_mb"], alpha=0.7)
    ax.annotate(r["model"], (r["cpu_fps"], r["map50_vehicle"]), textcoords="offset points", xytext=(8, 6))
ax.set_xlabel(f"CPU speed, FPS ({int(df['cpu_threads'].iloc[0])} threads, edge stand-in)")
ax.set_ylabel("Vehicle mAP@0.5")
ax.set_title("Accuracy vs speed (bubble size = model size)")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG / "accuracy_vs_speed.png", dpi=150); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for a, (col, title) in zip(axes, [("map50_vehicle", "Vehicle mAP@0.5"), ("cpu_fps", "CPU FPS"), ("size_mb", "Model size (MB)")]):
    a.bar(df["model"], df[col], color="#e8742c"); a.set_title(title); a.tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.savefig(FIG / "comparison_bars.png", dpi=150); plt.show()

In [ ]:
# Deployment feasibility: HEURISTIC screening. Edit the thresholds and justify them in your report.
# Colab's GPU/CPU are far faster than a Jetson Nano or Raspberry Pi, so these are proxy checks only.
FEAS = {"max_params_m": 10, "max_size_mb": 25, "min_cpu_fps": 10, "min_gpu_fps": 100}
f = df[["model", "params_m", "size_mb", "cpu_fps", "gpu_fps"]].copy()
f["small enough"] = (f.params_m <= FEAS["max_params_m"]) & (f.size_mb <= FEAS["max_size_mb"])
f["CPU fast enough (Pi proxy)"] = f.cpu_fps >= FEAS["min_cpu_fps"]
f["GPU fast enough (Jetson proxy)"] = f.gpu_fps.fillna(0) >= FEAS["min_gpu_fps"]
f["verdict"] = np.where(f["small enough"] & f["CPU fast enough (Pi proxy)"] & f["GPU fast enough (Jetson proxy)"],
                        "likely feasible", "needs optimisation (INT8 / smaller input / TensorRT)")
display(f.round(2)); print("thresholds:", FEAS)
f.round(3).to_csv(OUT / "feasibility.csv", index=False)
best_acc, fastest, smallest = df.loc[df.map50_vehicle.idxmax()], df.loc[df.cpu_fps.idxmax()], df.loc[df.size_mb.idxmin()]
print(f"Most accurate: {best_acc.model} ({best_acc.map50_vehicle:.3f} vehicle mAP@0.5)")
print(f"Fastest on CPU: {fastest.model} ({fastest.cpu_fps:.1f} FPS)")
print(f"Smallest: {smallest.model} ({smallest.size_mb:.1f} MB)")

## 6. Side-by-side detection images for the slides

In [ ]:
def draw(img, b, s, l, thr=0.4):
    out = img.copy()
    for (x1, y1, x2, y2), sc, c in zip(b, s, l):
        if sc < thr:
            continue
        cv2.rectangle(out, (int(x1), int(y1)), (int(x2), int(y2)), (232, 116, 44), 2)
        cv2.putText(out, f"{names.get(int(c), c)} {sc:.2f}", (int(x1), max(12, int(y1) - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return out

preds = {}
for tag in ("yolov8n", "yolov9t"):
    if (WTS / f"{tag}.pt").exists():
        preds[NAMES[tag]] = make_yolo_predictor(WTS / f"{tag}.pt", 0 if torch.cuda.is_available() else "cpu", 0.25)
if (WTS / "ssdlite.pt").exists():
    m = build_ssd(NC, SSD_SIZE, pretrained=False)
    m.load_state_dict(torch.load(WTS / "ssdlite.pt", map_location="cpu"))
    preds[NAMES["ssdlite"]] = make_ssd_predictor(m, "cuda" if torch.cuda.is_available() else "cpu", 0.25)

picks = random.Random(3).sample(val_imgs, 3)
fig, axes = plt.subplots(len(preds), 3, figsize=(18, 3.4 * len(preds)), squeeze=False)
for i, (name, fn) in enumerate(preds.items()):
    for j, p in enumerate(picks):
        im = cv2.imread(p)
        axes[i][j].imshow(draw(im, *fn(im))[..., ::-1]); axes[i][j].axis("off")
        if j == 0:
            axes[i][j].set_title(name, loc="left", fontsize=14)
plt.tight_layout(); plt.savefig(FIG / "side_by_side_detections.png", dpi=130); plt.show()

In [ ]:
zip_path = shutil.make_archive("/content/adas_results", "zip", str(OUT))
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Everything is in:", OUT)

## If something goes wrong
- **Out of memory:** lower `BATCH` for that model in the config cell.
- **Colab disconnected:** reconnect, re-run the setup, config, helper, dataset and function cells, then run the model cell again. Finished models are skipped automatically (their result file exists).
- **KITTI download fails:** re-run the dataset cell. If it keeps failing, tell your team so you can switch to a Kaggle copy.
- **Numbers differ slightly between runs:** normal. Always compare models run on the same GPU type.